# **Downstream latent-space experiments**

Model-agnostic probes that show whether each trained VAE's latent space is *ready for a diffusion model*, **without** training an LDM. Every model implements the same `encode_latents(x) -> list[Tensor]` / `decode_latents(list[Tensor]) -> (B,H,W,C)` contract, so all four run through the exact same code path.

**Experiment 1 — Latent noise-injection robustness.**  Encode a clean batch, add `N(0, sigma^2)` noise directly to the latents at `sigma in {0, 0.1, 0.5, 1.0}`, decode, and score reconstruction quality (MSE / SAM / PSNR / SSIM) vs the clean input. A robust manifold degrades gracefully; a fragile one collapses by sigma=0.5.

**Experiment 2 — Chemical interpolation smoothness.**  For two endpoint samples, linearly interpolate `z_mix = alpha*z_A + (1-alpha)*z_B` for `alpha in [0,1]`, decode, and track a single pixel's spectrum. *Jaggedness* = mean L2 of the second difference of decoded spectra along alpha (lower = smoother chemical transition = more generative-ready).

Self-contained (Kaggle/Colab-ready): all four model definitions, loss/metric primitives, dataloader, and experiment code live inside this notebook — no imports from the `PRISM` repo.

## **Config**

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path as _Path
import yaml as _yaml

# --------------------------------------------------------------------------
# Datasets in the ablation. Edit DATA_ROOTS to point at your processed patches
# (each root should contain <scene>/<split>/patch_*.npy). On Kaggle/Colab these
# will be /kaggle/input/... paths; locally they default to data/processed/<DS>.
# --------------------------------------------------------------------------
DATASETS = {
    "IIRS":   {"input_channels": 256},
    "M3":     {"input_channels": 84},
    "AVIRIS": {"input_channels": 424},
}

DATA_ROOTS = {
    "IIRS":   "data/processed/IIRS",
    "M3":     "data/processed/M3",
    "AVIRIS": "data/processed/AVIRIS",
}

# Where trained checkpoints are written: CKPT_ROOT/<DATASET>/<name>.pt
CKPT_ROOT = "model"

# Where the per-dataset hyperparam YAMLs live (repo-root-relative).
HYPERPARAM_CONFIG_DIR = "utils/hyperparam_configs"


@dataclass
class Settings:
    input_height: int = 64
    input_width: int = 64
    input_channels: int = 256          # overridden per dataset via make_settings()

    # training
    batch_size: int = 4
    num_workers: int = 4
    epochs: int = 20
    lr: float = 5e-3
    beta: float = 1e-3
    lambda_physics: float = 0.3

    # spatial branch
    reduced_dims: int = 32
    latent_dim: int = 256
    n_2D_conv_blocks: int = 4
    conv2D_kernel_size: int = 3
    conv_output_c: int = field(init=False)
    conv_output_h: int = field(init=False)
    conv_output_w: int = field(init=False)

    # spectral branch
    spectral_n_1D_conv_blocks: int = 2
    spectral_conv1D_kernel_size: int = 4
    spectral_latent_dim: int = 128
    spectral_linear_expansion_dim: int = field(init=False)
    spectral_transpose_c: int = field(init=False)
    spectral_transpose_l: int = field(init=False)

    # Baseline capacity knobs (overridden per-dataset by hyperparam YAML so each
    # baseline matches vae-our's param count at that dataset). IIRS defaults.
    vae_standard_base_ch: int = 134
    vae_standard_n_down: int = 3
    vae_standard_latent_ch: int = 16

    vae_3d_base_ch: int = 78
    vae_3d_n_down: int = 3
    vae_3d_latent_ch: int = 8

    vae_1d_hidden_dims: tuple = (4224, 2112, 1056)
    vae_1d_latent_dim: int = 32

    def __post_init__(self):
        self.conv_output_c = self.reduced_dims * (2 ** self.n_2D_conv_blocks)
        self.conv_output_h = self.input_height // (2 ** self.n_2D_conv_blocks)
        self.conv_output_w = self.input_width // (2 ** self.n_2D_conv_blocks)
        self.spectral_transpose_c = self.input_channels * (2 ** (self.spectral_n_1D_conv_blocks - 1))
        self.spectral_transpose_l = self.input_channels // (2 ** self.spectral_n_1D_conv_blocks)
        self.spectral_linear_expansion_dim = self.spectral_transpose_c * self.spectral_transpose_l


def make_settings(dataset):
    """Return a Settings whose band count matches the dataset."""
    return Settings(input_channels=DATASETS[dataset]["input_channels"])


# --------------------------------------------------------------------------
# Per-dataset hyperparam loader — mirrors utils/hyperparams.py but re-implemented
# locally so the notebook stays self-contained.
# --------------------------------------------------------------------------
_HP_SETTINGS_FIELDS = {
    "batch_size", "num_workers",
    "vae_standard_base_ch", "vae_standard_n_down", "vae_standard_latent_ch",
    "vae_3d_base_ch", "vae_3d_n_down", "vae_3d_latent_ch",
    "vae_1d_hidden_dims", "vae_1d_latent_dim",
    # notebook Settings also carries these as fields (unlike utils/config.py Settings):
    "epochs", "lr", "beta", "lambda_physics",
}
_HP_OPTIMIZATION_FIELDS = {"seed", "weight_decay", "early_stopping_patience"}
_HP_ALLOWED = _HP_SETTINGS_FIELDS | _HP_OPTIMIZATION_FIELDS


def load_hyperparams(dataset, config_dir=HYPERPARAM_CONFIG_DIR):
    """Load hyperparam-config-<DATASET>.yaml -> dict. Empty dict if missing."""
    path = _Path(config_dir) / f"hyperparam-config-{dataset.upper()}.yaml"
    if not path.exists():
        return {}
    with open(path) as f:
        raw = _yaml.safe_load(f) or {}
    unknown = set(raw) - _HP_ALLOWED
    if unknown:
        raise ValueError(f"{path}: unknown hyperparam keys: {sorted(unknown)}")
    return raw


def apply_hyperparams(settings, hp):
    """Mutate `settings` in place for whitelisted fields present in `hp`."""
    for key in _HP_SETTINGS_FIELDS & set(hp):
        value = hp[key]
        if key == "vae_1d_hidden_dims" and isinstance(value, list):
            value = tuple(value)
        setattr(settings, key, value)


# Global settings object the branch classes read from. Reassigned per dataset
# inside the training loop (rebuild the model after reassigning).
settings = make_settings("IIRS")


## **Imports**

In [ ]:
import json
from collections import OrderedDict
from pathlib import Path
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Conv2d, ConvTranspose2d
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## **Loss and metrics**

In [ ]:
def spectral_angle_mapper_loss(y_true, y_pred):
    """Differentiable physics prior: mean spectral angle (radians)."""
    dot = torch.sum(y_true * y_pred, dim=-1)
    nt = torch.sqrt(torch.sum(y_true ** 2, dim=-1) + 1e-8)
    npd = torch.sqrt(torch.sum(y_pred ** 2, dim=-1) + 1e-8)
    cos = torch.clamp(dot / (nt * npd + 1e-8), -1.0 + 1e-8, 1.0 - 1e-8)
    return torch.mean(torch.acos(cos))


def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())


def compute_mse(img1, img2):
    return F.mse_loss(img1, img2).item()


def compute_psnr(img1, img2, data_range=1.0):
    mse = F.mse_loss(img1, img2)
    if mse == 0:
        return float("inf")
    return (20 * torch.log10(torch.tensor(data_range).to(img1.device)) - 10 * torch.log10(mse)).item()


def compute_ssim(img1, img2, data_range=1.0, window_size=11):
    if img1.dim() == 4 and img1.shape[-1] not in [img1.shape[1], img1.shape[2]]:
        img1 = img1.permute(0, 3, 1, 2); img2 = img2.permute(0, 3, 1, 2)
    channels = img1.shape[1]

    def gaussian(w, sigma):
        g = torch.exp(torch.tensor([-(x - w // 2) ** 2 / (2 * sigma ** 2) for x in range(w)]))
        return g / g.sum()

    _1d = gaussian(window_size, 1.5).unsqueeze(1).to(img1.device)
    _2d = _1d.mm(_1d.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2d.expand(channels, 1, window_size, window_size).contiguous()
    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channels)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channels)
    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    s1 = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channels) - mu1_sq
    s2 = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channels) - mu2_sq
    s12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channels) - mu1_mu2
    c1 = (0.01 * data_range) ** 2; c2 = (0.03 * data_range) ** 2
    ssim = ((2 * mu1_mu2 + c1) * (2 * s12 + c2)) / ((mu1_sq + mu2_sq + c1) * (s1 + s2 + c2))
    return ssim.mean().item()

## **Data loader**

In [ ]:
class HSIPatchDataset(Dataset):
    """All .npy patches for a split, max-normalized to [0, 1] on the fly."""
    def __init__(self, processed_root, split):
        assert split in ("train", "valid", "test")
        self.patch_files: List[Path] = sorted(Path(processed_root).glob(f"**/{split}/*.npy"))

    def __len__(self):
        return len(self.patch_files)

    def __getitem__(self, idx):
        patch = np.load(self.patch_files[idx], mmap_mode="r").astype(np.float32)
        m = patch.max()
        if m > 0:
            patch = patch / m
        return torch.from_numpy(patch)


def build_dataloader(processed_root, split, batch_size=None, shuffle=True, num_workers=None, pin_memory=True):
    ds = HSIPatchDataset(processed_root, split)
    return DataLoader(
        ds,
        batch_size=batch_size or settings.batch_size,
        shuffle=shuffle,
        num_workers=num_workers if num_workers is not None else settings.num_workers,
        pin_memory=pin_memory,
        drop_last=(split == "train"),
    )

## **Model — vae-our (Dual-Stream PI-VAE)**

Two independent VAE streams reconstruct the full cube (spatial + spectral) and a learned linear layer late-fuses the two reconstructions. `encode_latents` returns `[z_spatial (B, latent_dim), z_spectral (B, spectral_latent_dim, H, W)]`.

In [ ]:
class SpatialEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.pixel_reduction = Conv2d(settings.input_channels, settings.reduced_dims, kernel_size=1)
        in_c = settings.reduced_dims; out_c = 2 * in_c
        layers = []
        for _ in range(settings.n_2D_conv_blocks):
            layers += [Conv2d(in_c, out_c, settings.conv2D_kernel_size, stride=2, padding=1), nn.ReLU()]
            in_c = out_c; out_c *= 2
        self.conv2D_block = nn.Sequential(*layers)
        self.flatten = nn.Flatten()
        self.linear = nn.LazyLinear(2 * settings.latent_dim)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.pixel_reduction(x)
        x = self.conv2D_block(x)
        x = self.flatten(x)
        return self.linear(x)


class SpatialDecoder(nn.Module):
    def __init__(self, conv_output_c, conv_output_h, conv_output_w):
        super().__init__()
        self.conv_out_c, self.conv_out_h, self.conv_out_w = conv_output_c, conv_output_h, conv_output_w
        self.linear = nn.Linear(settings.latent_dim, conv_output_c * conv_output_h * conv_output_w)
        in_c = conv_output_c; layers = []
        for _ in range(settings.n_2D_conv_blocks):
            out_c = in_c // 2
            layers += [nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1), nn.ReLU()]
            in_c = out_c
        self.transposeconv2D_block = nn.Sequential(*layers)
        self.pixel_expansion = Conv2d(settings.reduced_dims, settings.input_channels, kernel_size=1)

    def forward(self, z):
        B = z.shape[0]
        x = self.linear(z).view(B, self.conv_out_c, self.conv_out_h, self.conv_out_w)
        x = self.transposeconv2D_block(x)
        x = self.pixel_expansion(x)
        return x.permute(0, 2, 3, 1)


class SpatialEncoderDecoder(nn.Module):
    def __init__(self, conv_output_c, conv_output_h, conv_output_w):
        super().__init__()
        self.encoder = SpatialEncoder()
        self.decoder = SpatialDecoder(conv_output_c, conv_output_h, conv_output_w)


class SpectralEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        in_c = 1; out_c = settings.input_channels
        layers = []
        for _ in range(settings.spectral_n_1D_conv_blocks):
            layers += [Conv2d(in_c, out_c, kernel_size=(settings.spectral_conv1D_kernel_size, 1),
                              stride=(2, 1), padding=(1, 0)), nn.ReLU()]
            in_c = out_c; out_c *= 2
        self.conv1D_block = nn.Sequential(*layers)
        self.latent_proj = Conv2d(settings.spectral_linear_expansion_dim,
                                  2 * settings.spectral_latent_dim, kernel_size=1)

    def forward(self, x):
        batch, h, w, c = x.shape
        x = x.permute(0, 3, 1, 2).reshape(batch, 1, c, h * w)
        x = self.conv1D_block(x)
        x = x.reshape(batch, x.shape[1] * x.shape[2], h, w)
        return self.latent_proj(x)


class SpectralDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.latent_expand = Conv2d(settings.spectral_latent_dim,
                                    settings.spectral_linear_expansion_dim, kernel_size=1)
        in_c = settings.spectral_transpose_c; layers = []
        for i in range(settings.spectral_n_1D_conv_blocks):
            is_last = i == (settings.spectral_n_1D_conv_blocks - 1)
            out_c = 1 if is_last else in_c // 2
            layers.append(ConvTranspose2d(in_c, out_c, kernel_size=(settings.spectral_conv1D_kernel_size, 1),
                                          stride=(2, 1), padding=(1, 0)))
            if not is_last:
                layers.append(nn.ReLU())
            in_c = out_c
        self.transpose_block = nn.Sequential(*layers)

    def forward(self, z):
        batch, c, h, w = z.shape
        x = self.latent_expand(z)
        x = x.reshape(batch, settings.spectral_transpose_c, settings.spectral_transpose_l, h * w)
        x = self.transpose_block(x)
        x = x.squeeze(1).reshape(batch, settings.input_channels, h, w)
        return x.permute(0, 2, 3, 1)


class SpectralEncoderDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = SpectralEncoder()
        self.decoder = SpectralDecoder()


class HSI_DualStream_PI_VAE(nn.Module):
    """Dual-stream late-fusion physics-informed VAE (vae-our)."""
    def __init__(self):
        super().__init__()
        self.spatial_stream = SpatialEncoderDecoder(
            settings.conv_output_c, settings.conv_output_h, settings.conv_output_w)
        self.spectral_stream = SpectralEncoderDecoder()
        self.fusion_layer = nn.Linear(settings.input_channels * 2, settings.input_channels)

    def reparameterize(self, z_features):
        mu, logvar = torch.chunk(z_features, 2, dim=1)
        logvar = torch.clamp(logvar, min=-30.0, max=20.0)
        std = torch.exp(0.5 * logvar); eps = torch.randn_like(std)
        return mu + eps * std, mu, logvar

    def forward(self, x):
        sf = self.spatial_stream.encoder(x)
        z_s, mu_s, logvar_s = self.reparameterize(sf)
        recon_s = self.spatial_stream.decoder(z_s)
        pf = self.spectral_stream.encoder(x)
        z_p, mu_p, logvar_p = self.reparameterize(pf)
        recon_p = self.spectral_stream.decoder(z_p)
        combined = torch.cat([recon_s, recon_p], dim=-1)
        recon_final = torch.sigmoid(self.fusion_layer(combined))
        return recon_final, recon_s, recon_p, mu_s, logvar_s, mu_p, logvar_p

    @torch.no_grad()
    def encode_latents(self, x):
        mu_s, _ = torch.chunk(self.spatial_stream.encoder(x), 2, dim=1)
        mu_p, _ = torch.chunk(self.spectral_stream.encoder(x), 2, dim=1)
        return [mu_s, mu_p]

    @torch.no_grad()
    def decode_latents(self, latents):
        z_s, z_p = latents
        recon_s = self.spatial_stream.decoder(z_s)
        recon_p = self.spectral_stream.decoder(z_p)
        combined = torch.cat([recon_s, recon_p], dim=-1)
        return torch.sigmoid(self.fusion_layer(combined))

## **Model — vae-standard (Baseline A: 2D Spatial)**

AutoencoderKL-style 2D-conv VAE that treats the cube as a thick RGB image. `encode_latents` returns `[mu (B, 16, 8, 8)]`.

In [ ]:
class VAE_Standard(nn.Module):
    def __init__(self):
        super().__init__()
        c = settings.input_channels
        base_ch = settings.vae_standard_base_ch
        n_down = settings.vae_standard_n_down
        latent_ch = settings.vae_standard_latent_ch
        self.latent_ch = latent_ch
        enc = [nn.Conv2d(c, base_ch, 3, 1, 1), nn.ReLU()]
        in_c = base_ch
        for _ in range(n_down):
            enc += [nn.Conv2d(in_c, in_c * 2, 4, 2, 1), nn.ReLU()]; in_c *= 2
        enc.append(nn.Conv2d(in_c, 2 * latent_ch, 1))
        self.encoder = nn.Sequential(*enc)
        dec = [nn.Conv2d(latent_ch, in_c, 1), nn.ReLU()]
        for _ in range(n_down):
            dec += [nn.ConvTranspose2d(in_c, in_c // 2, 4, 2, 1), nn.ReLU()]; in_c //= 2
        dec.append(nn.Conv2d(in_c, c, 3, 1, 1))
        self.decoder = nn.Sequential(*dec)

    @staticmethod
    def reparameterize(params):
        mu, logvar = torch.chunk(params, 2, dim=1)
        logvar = torch.clamp(logvar, -30.0, 20.0)
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std, mu, logvar

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        z, mu, logvar = self.reparameterize(self.encoder(x))
        recon = torch.sigmoid(self.decoder(z)).permute(0, 2, 3, 1)
        return recon, mu, logvar

    @torch.no_grad()
    def encode_latents(self, x):
        mu, _ = torch.chunk(self.encoder(x.permute(0, 3, 1, 2)), 2, dim=1)
        return [mu]

    @torch.no_grad()
    def decode_latents(self, latents):
        return torch.sigmoid(self.decoder(latents[0])).permute(0, 2, 3, 1)


def compute_losses(model, x, beta, lambda_physics, use_physics=False):
    recon, mu, logvar = model(x)
    mse = F.mse_loss(recon, x)
    kld = kl_divergence(mu, logvar)
    sam = spectral_angle_mapper_loss(x, recon)
    loss = mse + beta * kld + (lambda_physics * sam if use_physics else 0.0)
    return loss, mse, sam, kld, recon


def build_model():
    return VAE_Standard()


## **Model — vae-3d-spatio-spectral (Baseline B: 3D)**

Fully-Conv3D VAE: the patch is a single-channel volume `(B, 1, C, H, W)`. `encode_latents` returns `[mu (B, 8, C, 8, 8)]`.

In [ ]:
_K, _S, _P = (3, 4, 4), (1, 2, 2), (1, 1, 1)


class VAE_3D_SpatioSpectral(nn.Module):
    def __init__(self):
        super().__init__()
        base_ch = settings.vae_3d_base_ch
        n_down = settings.vae_3d_n_down
        latent_ch = settings.vae_3d_latent_ch
        self.latent_ch = latent_ch
        enc = [nn.Conv3d(1, base_ch, 3, 1, 1), nn.ReLU()]
        in_c = base_ch
        for _ in range(n_down):
            enc += [nn.Conv3d(in_c, in_c * 2, _K, _S, _P), nn.ReLU()]; in_c *= 2
        enc.append(nn.Conv3d(in_c, 2 * latent_ch, 1))
        self.encoder = nn.Sequential(*enc)
        dec = [nn.Conv3d(latent_ch, in_c, 1), nn.ReLU()]
        for _ in range(n_down):
            dec += [nn.ConvTranspose3d(in_c, in_c // 2, _K, _S, _P), nn.ReLU()]; in_c //= 2
        dec.append(nn.Conv3d(in_c, 1, 3, 1, 1))
        self.decoder = nn.Sequential(*dec)

    @staticmethod
    def reparameterize(params):
        mu, logvar = torch.chunk(params, 2, dim=1)
        logvar = torch.clamp(logvar, -30.0, 20.0)
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std, mu, logvar

    def forward(self, x):
        vol = x.permute(0, 3, 1, 2).unsqueeze(1)
        z, mu, logvar = self.reparameterize(self.encoder(vol))
        recon = torch.sigmoid(self.decoder(z)).squeeze(1).permute(0, 2, 3, 1)
        return recon, mu, logvar

    @torch.no_grad()
    def encode_latents(self, x):
        mu, _ = torch.chunk(self.encoder(x.permute(0, 3, 1, 2).unsqueeze(1)), 2, dim=1)
        return [mu]

    @torch.no_grad()
    def decode_latents(self, latents):
        return torch.sigmoid(self.decoder(latents[0])).squeeze(1).permute(0, 2, 3, 1)


## **Model — vae-1d-pixelwise (Baseline C: 1D MLP)**

Per-pixel MLP VAE, folding `(B,H,W)` into the batch dim so each pixel is encoded independently. `encode_latents` returns `[mu (B, H, W, 32)]`.

In [ ]:
class VAE_1D_Pixelwise(nn.Module):
    def __init__(self):
        super().__init__()
        c = settings.input_channels
        hidden_dims = tuple(settings.vae_1d_hidden_dims)
        latent_dim = settings.vae_1d_latent_dim
        self.latent_dim = latent_dim
        enc, in_f = [], c
        for h in hidden_dims:
            enc += [nn.Linear(in_f, h), nn.ReLU()]; in_f = h
        enc.append(nn.Linear(in_f, 2 * latent_dim))
        self.encoder = nn.Sequential(*enc)
        dec, in_f = [], latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(in_f, h), nn.ReLU()]; in_f = h
        dec.append(nn.Linear(in_f, c))
        self.decoder = nn.Sequential(*dec)

    @staticmethod
    def reparameterize(params):
        mu, logvar = torch.chunk(params, 2, dim=-1)
        logvar = torch.clamp(logvar, -30.0, 20.0)
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std, mu, logvar

    def forward(self, x):
        b, h, w, c = x.shape
        z, mu, logvar = self.reparameterize(self.encoder(x.reshape(b * h * w, c)))
        recon = torch.sigmoid(self.decoder(z)).reshape(b, h, w, c)
        mu = mu.reshape(b, h, w, self.latent_dim)
        logvar = logvar.reshape(b, h, w, self.latent_dim)
        return recon, mu, logvar

    @torch.no_grad()
    def encode_latents(self, x):
        b, h, w, c = x.shape
        mu, _ = torch.chunk(self.encoder(x.reshape(b * h * w, c)), 2, dim=-1)
        return [mu.reshape(b, h, w, self.latent_dim)]

    @torch.no_grad()
    def decode_latents(self, latents):
        z = latents[0]; b, h, w, _ = z.shape
        recon = torch.sigmoid(self.decoder(z.reshape(b * h * w, self.latent_dim)))
        return recon.reshape(b, h, w, settings.input_channels)


## **Registry & ckpt helpers**

In [ ]:
MODEL_BUILDERS = {
    "vae-our":                HSI_DualStream_PI_VAE,
    "vae-standard":           VAE_Standard,
    "vae-3d-spatio-spectral": VAE_3D_SpatioSpectral,
    "vae-1d-pixelwise":       VAE_1D_Pixelwise,
}

# vae-our has SAM baked into its loss; there is no "standard" variant on disk.
PHYSICS_ONLY = {"vae-our"}


def checkpoint_name(model_name, loss_type):
    """Filename convention: vae-our.pt (physics-only) or <model>_<loss>.pt."""
    if model_name in PHYSICS_ONLY:
        return f"{model_name}.pt"
    return f"{model_name}_{loss_type}.pt"


def strip_module_prefix(state):
    out = OrderedDict()
    for k, v in state.items():
        out[k[len("module."):] if k.startswith("module.") else k] = v
    return out


def load_checkpoint(model_name, ckpt_file, device):
    """Instantiate a model by name, warm it up with a dummy batch, and load weights."""
    model = MODEL_BUILDERS[model_name]().to(device)
    with torch.no_grad():   # materialize LazyLinear layers (vae-our)
        model(torch.randn(2, settings.input_height, settings.input_width,
                          settings.input_channels, device=device))
    ckpt = torch.load(ckpt_file, map_location=device)
    state = ckpt.get("model_state_dict", ckpt)
    model.load_state_dict(strip_module_prefix(state))
    model.eval()
    return model

## **Downstream experiment helpers**

In [ ]:
def add_latent_noise(latents, sigma, generator=None):
    """Return a new latent list with N(0, sigma^2) noise added to every tensor."""
    out = []
    for z in latents:
        if sigma > 0:
            noise = torch.randn(z.shape, device=z.device, dtype=z.dtype, generator=generator)
            out.append(z + sigma * noise)
        else:
            out.append(z.clone())
    return out


def lerp_latents(latents_a, latents_b, alpha):
    """Elementwise z_mix = alpha*z_A + (1-alpha)*z_B for each tensor in the list."""
    return [alpha * za + (1.0 - alpha) * zb for za, zb in zip(latents_a, latents_b)]


def slice_latents(latents, idx):
    """Keep a single batch element (index idx), dim 0 preserved."""
    return [z[idx:idx + 1] for z in latents]


@torch.no_grad()
def noise_injection(model, x, sigmas, generator=None):
    """
    Encode x, inject latent noise at each sigma, decode, and score vs clean x.

    Returns list of dicts: {sigma, mse, sam, psnr, ssim}.
    """
    latents = model.encode_latents(x)
    results = []
    for sigma in sigmas:
        noisy = add_latent_noise(latents, sigma, generator=generator)
        recon = model.decode_latents(noisy)
        results.append({
            "sigma": float(sigma),
            "mse":   compute_mse(x, recon),
            "sam":   spectral_angle_mapper_loss(x, recon).item(),
            "psnr":  compute_psnr(x, recon),
            "ssim":  compute_ssim(x, recon),
        })
    return results


def _decoded_pixel_spectrum(recon, pixel):
    r, c = pixel
    return recon[0, r, c].detach().cpu().numpy()


@torch.no_grad()
def interpolation_smoothness(model, x, idx_a, idx_b, n_alpha, pixel):
    """
    Walk alpha 0->1 between two samples' latents, decode each mix, read the
    spectrum at ``pixel``, and score smoothness.

    jaggedness = mean L2 of the 2nd difference of decoded spectra along alpha
                 (lower = smoother = more generative-ready).
    path_length = sum of consecutive step magnitudes (context only).
    """
    latents = model.encode_latents(x)
    la = slice_latents(latents, idx_a)
    lb = slice_latents(latents, idx_b)

    alphas = np.linspace(0.0, 1.0, n_alpha)
    spectra = []
    for alpha in alphas:
        mix = lerp_latents(la, lb, float(alpha))
        recon = model.decode_latents(mix)
        spectra.append(_decoded_pixel_spectrum(recon, pixel))
    spectra = np.stack(spectra, axis=0)          # (n_alpha, C)

    if n_alpha >= 3:
        second_diff = spectra[:-2] - 2.0 * spectra[1:-1] + spectra[2:]
        jaggedness = float(np.mean(np.linalg.norm(second_diff, axis=-1)))
    else:
        jaggedness = float("nan")

    steps = np.linalg.norm(np.diff(spectra, axis=0), axis=-1)
    path_length = float(np.sum(steps))

    return {
        "alphas": alphas.tolist(),
        "spectra": spectra,
        "jaggedness": jaggedness,
        "path_length": path_length,
    }

## **Run — configuration**

Edit `DS`, `MODELS`, and (for the baselines) `LOSS` here. The clean batch is drawn once from the test split and reused across every model, so the comparison is fair.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DS = "IIRS"                                                  # "IIRS" | "M3" | "AVIRIS"
MODELS = ["vae-our", "vae-standard", "vae-3d-spatio-spectral", "vae-1d-pixelwise"]
LOSS = "physics"                                             # "standard" | "physics" (baselines only; vae-our is always physics)
SPLIT = "test"                                               # "train" | "valid" | "test"
BATCH_SIZE = 8                                               # patches used for the experiments

SIGMAS = (0.0, 0.1, 0.5, 1.0)
IDX_A, IDX_B, N_ALPHA = 0, 1, 11
PIXEL = (32, 32)                                             # (row, col) whose spectrum is tracked
SEED = 42

SAVE_PLOTS = True
OUT_DIR = Path("visualisations/downstream") / DS
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Configure the global settings for this dataset (drives model dims / dataloader);
# apply per-dataset hyperparam overrides so baseline widths match training-time.
settings = make_settings(DS)
apply_hyperparams(settings, load_hyperparams(DS))
globals()["settings"] = settings
torch.manual_seed(SEED)

# One fixed clean batch shared across all models.
loader = build_dataloader(DATA_ROOTS[DS], SPLIT, batch_size=BATCH_SIZE, shuffle=False)
x = next(iter(loader)).to(device)
assert x.shape[0] >= 2, "Need >=2 samples in the batch for interpolation endpoints."
IDX_A = min(IDX_A, x.shape[0] - 1)
IDX_B = min(IDX_B, x.shape[0] - 1)

print(f"dataset  : {DS}  (C={settings.input_channels})")
print(f"models   : {MODELS}")
print(f"split    : {SPLIT}  |  batch: {x.shape[0]}")
print(f"sigmas   : {SIGMAS}")
print(f"device   : {device}")

## **Run — per-model experiments**

In [ ]:
all_results = []
for model_name in MODELS:
    loss_for_ckpt = "physics" if model_name in PHYSICS_ONLY else LOSS
    ckpt_file = Path(CKPT_ROOT) / DS / checkpoint_name(model_name, loss_for_ckpt)
    if not ckpt_file.exists():
        print(f"[skip] {model_name}: checkpoint not found ({ckpt_file})")
        continue

    model = load_checkpoint(model_name, ckpt_file, device)

    # Fresh generator per model so each sees the same noise draw sequence.
    gen = torch.Generator(device=device).manual_seed(SEED)
    noise = noise_injection(model, x, SIGMAS, generator=gen)
    interp = interpolation_smoothness(model, x, IDX_A, IDX_B, N_ALPHA, PIXEL)

    all_results.append({
        "model": model_name,
        "loss":  loss_for_ckpt,
        "ckpt":  str(ckpt_file),
        "noise": noise,
        "interp": interp,
    })
    print(f"[ok]   {model_name}  ({ckpt_file.name})")

assert all_results, "No models could be evaluated (no checkpoints found)."

## **Experiment 1 — Latent noise-injection robustness (table)**

In [ ]:
print("=" * 72)
print(f" EXPERIMENT 1 - Latent noise-injection robustness - {DS}")
print("=" * 72)
for metric in ("psnr", "ssim", "sam"):
    arrow = "higher=better" if metric in ("psnr", "ssim") else "lower=better, rad"
    print(f"\n {metric.upper()}  ({arrow}) by sigma:")
    header = f"  {'model':24s}" + "".join(f"{f'sigma={s:g}':>12s}" for s in SIGMAS)
    print(header)
    for r in all_results:
        by_sigma = {d["sigma"]: d[metric] for d in r["noise"]}
        row = f"  {r['model']:24s}" + "".join(f"{by_sigma.get(float(s), float('nan')):12.4f}" for s in SIGMAS)
        print(row)

## **Experiment 2 — Chemical interpolation smoothness (table)**

In [ ]:
print("=" * 72)
print(f" EXPERIMENT 2 - Chemical interpolation smoothness - {DS}, pixel {PIXEL}")
print("=" * 72)
print(f"  {'model':24s}{'jaggedness (lower=smoother)':>32s}{'path_length':>16s}")
for r in all_results:
    it = r["interp"]
    print(f"  {r['model']:24s}{it['jaggedness']:32.5f}{it['path_length']:16.5f}")
print("\n  jaggedness = mean L2 of 2nd difference of decoded spectra along alpha")
print("  (lower = smoother chemical transition = more generative-ready manifold)")

## **Plots — noise-vs-PSNR and interpolation spectra**

In [ ]:
# Plot 1: PSNR-vs-sigma degradation curves (all models on one axis).
plt.figure(figsize=(7, 5))
for r in all_results:
    s = [d["sigma"] for d in r["noise"]]
    p = [d["psnr"] for d in r["noise"]]
    plt.plot(s, p, marker="o", label=r["model"])
plt.xlabel("latent noise sigma"); plt.ylabel("PSNR (dB)")
plt.title(f"Latent noise-injection robustness - {DS}")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
if SAVE_PLOTS:
    plt.savefig(OUT_DIR / "noise_robustness_psnr.png", dpi=150)
plt.show()

# Plot 2: interpolation spectra (one subplot per model) at the chosen pixel.
n = len(all_results)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
for ax, r in zip(axes[0], all_results):
    it = r["interp"]
    spectra = np.asarray(it["spectra"])
    for k, alpha in enumerate(it["alphas"]):
        ax.plot(spectra[k], color=plt.cm.viridis(alpha), alpha=0.8)
    ax.set_title(f"{r['model']}\njagged={it['jaggedness']:.4f}")
    ax.set_xlabel("band"); ax.set_ylabel("reflectance (norm.)")
plt.suptitle(f"Chemical interpolation (alpha: 0 -> 1) - {DS}, pixel {PIXEL}")
plt.tight_layout()
if SAVE_PLOTS:
    plt.savefig(OUT_DIR / "interpolation_spectra.png", dpi=150)
plt.show()

## **Save metrics JSON**

In [ ]:
serializable = []
for r in all_results:
    rc = {k: v for k, v in r.items() if k != "interp"}
    it = dict(r["interp"])
    it["spectra"] = np.asarray(it["spectra"]).tolist()
    rc["interp"] = it
    serializable.append(rc)

out_json = OUT_DIR / "downstream_results.json"
out_json.write_text(json.dumps(serializable, indent=2))
print(f"Saved metrics JSON to {out_json}")